In [1]:
import json
import time
import os
import pandas as pd
from tavily import TavilyClient
import asyncio
from tavily import AsyncTavilyClient

# Initialize Tavily
#tavily_client = TavilyClient(api_key="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") #4k
tavily_client = TavilyClient(api_key="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") #1k

Links importantes:

[ base officielle FINESS ](https://www.data.gouv.fr/datasets/finess-extraction-du-fichier-des-etablissements)

In [2]:
df = pd.read_csv(
    "../data/intermediate/Emergency Room Lookup.csv",
    sep=";",          # ← this is the key
    encoding="utf-8",
)

df = df.rename(columns={'id': 'record_id'})
df["comm"] = df["comm"].astype("Int64")
df["siret"] = df["siret"].astype("Int64")
df = df.drop(columns=["Unnamed: 0"])
df.head()

,record_id,finess,nom_etab_1,nom_etab_2,nom_etab_long,compl_etab,compl_distr,achemin,comm,dept,...,siret,code_ape,code_mft,lib_mft,code_sph,lib_sph,date_ouv,date_autor,date_maj,num_educ
0,010000024G1,010000024,CH DE FLEYRIAT,CH DE FLEYRIAT,CENTRE HOSPITALIER DE BOURG-EN-BRESSE FLEYRIAT,NaN,NaN,01440 VIRIAT,451,01,...,26010004500012,8610Z,3.0,ARS établissements Publics de santé dotation g...,1.0,Etablissement public de santé,1979-02-13,1979-02-13,2020-02-04,NaN
1,010000032G1,010000032,CH BUGEY SUD,CH BUGEY SUD,CENTRE HOSPITALIER BUGEY SUD,NaN,NaN,01300 BELLEY,34,01,...,26010003700068,8610Z,3.0,ARS établissements Publics de santé dotation g...,1.0,Etablissement public de santé,1901-01-01,1901-01-01,2021-07-07,NaN
2,010005239G1,010005239,CH DU HAUT BUGEY - GEOVREISSET,CH DU HAUT BUGEY - GEOVREISSET,CENTRE HOSPITALIER DU HAUT BUGEY - GEOVREISSET,NaN,NaN,01108 OYONNAX CEDEX,283,01,...,26011021800112,NaN,3.0,ARS établissements Publics de santé dotation g...,1.0,Etablissement public de santé,2007-04-20,2007-03-14,2018-01-12,NaN
3,010780195G1,010780195,CLINIQUE CONVERT,CLINIQUE CONVERT,CLINIQUE DOCTEUR CONVERT,NaN,NaN,01000 BOURG EN BRESSE,53,01,...,77220148900022,8610Z,7.0,ARS établissements de santé non financés dotat...,0.0,Non concerné,1956-11-16,1956-11-16,2021-10-18,NaN
4,010780203G1,010780203,HOPITAL PRIVE D AMBERIEU,HOPITAL PRIVE D'AMBERIEU,HOPITAL PRIVE D'AMBERIEU,NaN,NaN,01506 AMBERIEU EN BUGEY CEDEX,4,01,...,81157144700010,8610Z,7.0,ARS établissements de santé non financés dotat...,0.0,Non concerné,2001-09-03,1994-04-03,2024-01-30,NaN


In [3]:
# Create a summaary to review the structure of the dataset
summary = pd.DataFrame({
    'Unique Values': df.nunique(),
    'Data Type': df.dtypes,
    'Missing Values (Total)': df.isnull().sum(),
    'Missing Values (%)': (df.isnull().sum() / len(df)) * 100
})
# Sort by percentage of missing values
summary = summary.sort_values(by='Missing Values (%)', ascending=False)

print(summary)

                Unique Values Data Type  Missing Values (Total)  \
num_educ                   13    object                     701   
compl_distr                34    object                     673   
compl_etab                 51    object                     641   
num_voie                  155   float64                     176   
nom_etab_long             510    object                     106   
code_ape                    4    object                      70   
typ_voie                   21    object                      18   
lib_voie                  533    object                      13   
siret                     599     Int64                       9   
code_sph                    6   float64                       2   
lib_sph                     6    object                       2   
lib_mft                     9    object                       1   
lib_etab                    5    object                       1   
date_ouv                  312    object                       

In [4]:
df.shape

(715, 28)

In [5]:
sorted(df["dept"].dropna().unique())

['01',
 '02',
 '03',
 '04',
 '05',
 '06',
 '07',
 '08',
 '09',
 '10',
 '11',
 '12',
 '13',
 '14',
 '15',
 '16',
 '17',
 '18',
 '19',
 '21',
 '22',
 '23',
 '24',
 '25',
 '26',
 '27',
 '28',
 '29',
 '2A',
 '2B',
 '30',
 '31',
 '32',
 '33',
 '34',
 '35',
 '36',
 '37',
 '38',
 '39',
 '40',
 '41',
 '42',
 '43',
 '44',
 '45',
 '46',
 '47',
 '48',
 '49',
 '50',
 '51',
 '52',
 '53',
 '54',
 '55',
 '56',
 '57',
 '58',
 '59',
 '60',
 '61',
 '62',
 '63',
 '64',
 '65',
 '66',
 '67',
 '68',
 '69',
 '70',
 '71',
 '72',
 '73',
 '74',
 '75',
 '76',
 '77',
 '78',
 '79',
 '80',
 '81',
 '82',
 '83',
 '84',
 '85',
 '86',
 '87',
 '88',
 '89',
 '90',
 '91',
 '92',
 '93',
 '94',
 '95',
 '9A',
 '9B',
 '9C',
 '9D']

In [6]:
df["lib_sph"].value_counts()

lib_sph
Etablissement public de santé                                   554
Non concerné                                                    118
Etablissement de santé privé d'intérêt collectif                 29
indéterminé                                                       8
Etab de santé privé non lucratif, non déclar intérêt collect      3
PSPH par concession                                               1
Name: count, dtype: int64

In [7]:
df["nom_etab_1"].str.contains("CH").mean()

0.5692307692307692

In [8]:
def clean_long_name(name):
    """
    Strips administrative prefixes from long hospital names 
    to extract the specific location or site name.
    """
    if not name or pd.isna(name):
        return ""
    
    # List of prefixes to remove, ordered from longest to shortest
    # to avoid partial replacement issues.
    prefixes = [
        "CENTRE HOSPITALIER REGIONAL ET UNIVERSITAIRE DE ",
        "CENTRE HOSPITALIER UNIVERSITAIRE DE ",
        "CENTRE HOSPITALIER REGIONAL DE ",
        "CENTRE HOSPITALIER DE ",
        "CENTRE HOSPITALIER ",
        "CH ",
        "HOPITAL PRIVE DE ",
        "HOPITAL PRIVE D' ",
        "HOPITAL PRIVE ",
        "CLINIQUE DOCTEUR ",
        "CLINIQUE DU ",
        "CLINIQUE DE ",
        "CLINIQUE "
    ]
    
    clean = str(name).upper()
    
    for p in prefixes:
        if clean.startswith(p):
            clean = clean.replace(p, "", 1) # Only replace the first occurrence
            break # Once a prefix is matched and removed, we stop
            
    return clean.strip()

# Apply to your dataframe
# Use nom_etab_long when available, fall back to nom_etab if NaN
df['keywords_nom_etab'] = df.apply(
    lambda x: clean_long_name(x['nom_etab_long']) if pd.notna(x['nom_etab_long']) 
    else clean_long_name(x['nom_etab_1']), 
    axis=1
)
df["keywords_nom_etab"].isna().sum()

0

Let's keep just the short names of the hospitals 

In [9]:
df["ville"] = df["achemin"].str.replace(r"^\d{5}\s+", "", regex=True)
df_unique = df[["finess", "nom_etab_1", "nom_etab_long",'keywords_nom_etab', "ville", "record_id"]].reset_index(drop=True)
df_unique.head(10)

,finess,nom_etab_1,nom_etab_long,keywords_nom_etab,ville,record_id
0,010000024,CH DE FLEYRIAT,CENTRE HOSPITALIER DE BOURG-EN-BRESSE FLEYRIAT,BOURG-EN-BRESSE FLEYRIAT,VIRIAT,010000024G1
1,010000032,CH BUGEY SUD,CENTRE HOSPITALIER BUGEY SUD,BUGEY SUD,BELLEY,010000032G1
2,010005239,CH DU HAUT BUGEY - GEOVREISSET,CENTRE HOSPITALIER DU HAUT BUGEY - GEOVREISSET,DU HAUT BUGEY - GEOVREISSET,OYONNAX CEDEX,010005239G1
3,010780195,CLINIQUE CONVERT,CLINIQUE DOCTEUR CONVERT,CONVERT,BOURG EN BRESSE,010780195G1
4,010780203,HOPITAL PRIVE D AMBERIEU,HOPITAL PRIVE D'AMBERIEU,D'AMBERIEU,AMBERIEU EN BUGEY CEDEX,010780203G1
5,020000162,CH SAINT-QUENTIN,NaN,SAINT-QUENTIN,ST QUENTIN CEDEX,020000162A1
6,020000162,CH SAINT-QUENTIN,NaN,SAINT-QUENTIN,ST QUENTIN CEDEX,020000162P1
7,020000394,CH LAON,NaN,LAON,LAON CEDEX,020000394A1
8,020000394,CH LAON,NaN,LAON,LAON CEDEX,020000394P1
9,020000519,CH SOISSONS,NaN,SOISSONS,SOISSONS CEDEX,020000519A1


In [10]:
df_unique = df.drop_duplicates(subset="finess", keep="first")

In [11]:
print(len(df_unique["nom_etab_1"].unique().tolist()))

609


In [12]:
print(len(df_unique["keywords_nom_etab"].unique().tolist()))

603


In [13]:
print(len(df_unique["nom_etab_long"].unique().tolist()))

511


In [14]:
print(len(df_unique["ville"].unique().tolist()))

567


There are 609 unique FINESS codes. 

# General Search

In [15]:
# 1. Setup
# Define the output jsonl file
output_path_simple = "/Data/anahi_reyes/EDCD_data/raw_edcd_database_simple_full.jsonl"
folder_path = os.path.dirname(output_path_simple )

# Create folder if it does not exist
os.makedirs(folder_path, exist_ok=True)
print(f"Verified directory: {output_path_simple }")

Verified directory: /Data/anahi_reyes/EDCD_data/raw_edcd_database_simple_full.jsonl


In [ ]:
for index, row in df_unique.head(2).iterrows():
    finess = row["finess"]
    name = row["nom_etab_1"]
    keywords_nom_etab = row["keywords_nom_etab"]
    nom_etab_long = row["nom_etab_long"]

    try:
        query = f'("{name}" OR "{keywords_nom_etab}") urgences France ("fermeture temporaire" OR "grève" OR "SMUR" OR "communiqué de presse")'

        search = tavily_client.search(
            query=query,
            search_depth="advanced",
            topic="general", #ponerle news hace que regrese basura
            #chunks_per_source=3,
            include_raw_content=True,
            exact_match=True,
            max_results=10,
            country="france",
            start_date="2023-01-01",
            end_date="2025-12-31"
        )  # ← closing parenthesis added

        results = search.get('results', [])

        # Save ONE ROW PER SOURCE (atomic format)
        # Each source likely describes a different closure event
        for result in results:
            entry = {
                "finess": finess,
                "hospital_name": name,
                "nom_etab_long": nom_etab_long,
                "keywords_nom_etab": keywords_nom_etab,
                "source_url": result.get("url"),
                "title": result.get("title"),
                "content": result.get("content"),
                "score": result.get("score"),
                "retrieved_at": time.strftime("%Y-%m-%d %H:%M:%S")
            }

            with open(output_path_simple, "a", encoding="utf-8") as f:
                f.write(json.dumps(entry, ensure_ascii=False) + "\n")

        print(f"✅ {name}: {len(results)} sources found")
        time.sleep(1)

    except Exception as e:
        print(f"❌ Error with {name}: {str(e)}")


print(f"Deep context dataset saved in: {output_path_simple}")

✅ CH DE FLEYRIAT: 10 sources found
✅ CH BUGEY SUD: 10 sources found
Deep context dataset saved in: /Data/anahi_reyes/EDCD_data/raw_edcd_database_simple_full.jsonl


In [ ]:
for index, row in df_unique.head(2).iterrows():
    finess = row["finess"]
    name = row["nom_etab_1"]
    keywords_nom_etab = row["keywords_nom_etab"]
    nom_etab_long = row["nom_etab_long"]

    query=f'("{name}" OR "{keywords_nom_etab}") urgences France ("fermeture temporaire" OR "grève" OR "SMUR" OR "communiqué de presse")',

    search = tavily_client.search(
        query=query,
        search_depth="advanced",
        topic="news",
        #chunks_per_source=3,
        include_raw_content=True, # if you need to do regex extraction
        exact_match=True,
        max_results=10,         # ← keep high to catch multiple events
        country="france",
        start_date="2023-01-01",
        end_date="2025-12-31"
    

    results = search.get('results', [])
        # Save ONE ROW PER SOURCE (atomic format)
        # Each source likely describes a different closure event
        for result in results:
            entry = {
                "finess": finess,
                "hospital_name": name,
                "nom_etab_long": nom_etab_long,
                "keywords_nom_etab": keywords_nom_etab,
                "source_url": result.get("url"),
                "title": result.get("title"),
                "content": result.get("content"),
                "score": result.get("score"),
                "retrieved_at": time.strftime("%Y-%m-%d %H:%M:%S")
            }

            with open(output_path_simple, "a", encoding="utf-8") as f:
                f.write(json.dumps(entry, ensure_ascii=False) + "\n")

        print(f"✅ {name}: {len(results)} sources found")
        time.sleep(1)

    except Exception as e:
        print(f"❌ Error with {name}: {str(e)}")


print(f"Deep context dataset saved in: {output_path_simple}")

✅ CH DE FLEYRIAT: 10 sources found
✅ CH BUGEY SUD: 10 sources found
Deep context dataset saved in: /Data/anahi_reyes/EDCD_data/raw_edcd_database_simple_full.jsonl


In [24]:
# 1. Setup
# Define the output jsonl file
output_path_simple = "/Data/anahi_reyes/EDCD_data/raw_edcd_database_simple.jsonl"
folder_path = os.path.dirname(output_path_simple )

# Create folder if it does not exist
os.makedirs(folder_path, exist_ok=True)
print(f"Verified directory: {output_path_simple }")

Verified directory: /Data/anahi_reyes/EDCD_data/raw_edcd_database_simple.jsonl


In [30]:
for index, row in df_unique.head(2).iterrows():
    finess = row["finess"]
    name = row["nom_etab_1"]
    keywords_nom_etab = row["keywords_nom_etab"]
    nom_etab_long = row["nom_etab_long"]

    query = f'("{name}" OR "{keywords_nom_etab}") service des urgences fermeture régulation accès régulé'

    if len(query) > 400:
        query = f'"{keywords_nom_etab}" service des urgences fermeture régulation accès régulé'

    try:
        search = tavily_client.search(
            query=query,
            search_depth="advanced",
            topic="general",
            chunks_per_source=3,
            max_results=10,         # ← keep high to catch multiple events
            country="france",
            start_date="2022-06-01",
            end_date="2025-12-31"
        )

        results = search.get('results', [])

        # Save ONE ROW PER SOURCE (atomic format)
        # Each source likely describes a different closure event
        for result in results:
            entry = {
                "finess": finess,
                "hospital_name": name,
                "nom_etab_long": nom_etab_long,
                "keywords_nom_etab": keywords_nom_etab,
                "source_url": result.get("url"),
                "title": result.get("title"),
                "content": result.get("content"),
                "score": result.get("score"),
                "retrieved_at": time.strftime("%Y-%m-%d %H:%M:%S")
            }

            with open(output_path_simple, "a", encoding="utf-8") as f:
                f.write(json.dumps(entry, ensure_ascii=False) + "\n")

        print(f"✅ {name}: {len(results)} sources found")
        time.sleep(1)

    except Exception as e:
        print(f"❌ Error with {name}: {str(e)}")


print(f"Deep context dataset saved in: {output_path_simple}")

✅ CH DE FLEYRIAT: 10 sources found
✅ CH BUGEY SUD: 10 sources found
Deep context dataset saved in: /Data/anahi_reyes/EDCD_data/raw_edcd_database_simple.jsonl


In [28]:
# 2. Main Loop
for index, row in df_unique.sample(2).iterrows():
    finess = row["finess"]
    name = row["nom_etab_1"]
    keywords_nom_etab = row["keywords_nom_etab"]
    nom_etab_long = row["nom_etab_long"]
    ville = row["ville"]
    
    # Keep it short — use keywords 
    query = f'("{name}" OR "{keywords_nom_etab}") service des urgences fermeture régulation accès régulé'


    search = tavily_client.search(
        query=query,
        search_depth="advanced",
        topic="general",  # includes ALL source types
        chunks_per_source=3,
        #include_raw_content = True,
        max_results=10,
        country="france",
        start_date="2022-06-01",
        end_date="2025-12-31",
        ensure_ascii=False
    )
    
    # 3. Save
    entry = {
        "finess": finess,
        "hospital_name": name,
        "nom_etab_long": nom_etab_long,
        "keywords_nom_etab": keywords_nom_etab,
        "search": search.get('results', []),
        "retrieved_at": time.strftime("%Y-%m-%d %H:%M:%S")
        }

    with open(output_path_simple, "a", encoding="utf-8") as f:
        f.write(json.dumps(entry) + "\n")
        
    print(f"✅ Search Complete: {name}")
    time.sleep(1)

#    except Exception as e: # <--- Now properly follows the try block
#        print(f"❌ Error with {name}: {str(e)}")
    
print(f"Deep context dataset saved in: {output_path_simple}")

✅ Search Complete: CHITS CH GEORGE SAND
✅ Search Complete: CHU SITE FELIX GUYON (SAINT DENIS)
Deep context dataset saved in: /Data/anahi_reyes/EDCD_data/raw_edcd_database_simple.jsonl


# Detailed instructions

In [ ]:
# 1. Setup
# Define the output jsonl file
output_path = "/Data/anahi_reyes/EDCD_data/raw_edcd_database.jsonl"
folder_path = os.path.dirname(output_path)

# Create folder if it does not exist
os.makedirs(folder_path, exist_ok=True)
print(f"Verified directory: {output_path}")

Verified directory: /Data/anahi_reyes/EDCD_data/raw_edcd_database.jsonl


In [121]:
# We keep Official domains strict to ensure "Legal Truth"
official_domains = "site:ars.sante.fr OR site:prefectures-regions.gouv.fr OR site:sante.gouv.fr"

# 2. Main Loop
for index, row in df.iterrows():
    name = row["nom_etab"]
    clean_key = row["clean_key"]
    finess = row["finess"]
    record_id = row["record_id"]
    
    # --- STREAM 1: OFFICIAL (Strict) ---
    query_official = (
        f'("{name}" OR "{clean_key}") '
        f'(urgences OR "service des urgences") '
        f'(fermeture OR "régulation" OR "accès régulé") '
        f'({official_domains})'
    )

    # --- STREAM 2: UNOFFICIAL (Open & Global) ---
    # No "site:" restrictions here. We add "communique de presse" and "France" to avoid generic international results.
    query_unofficial = (
        f'("{name}" OR "{clean_key}") '
        f'urgences France '
        f'("fermeture temporaire" OR "grève" OR "SMUR" OR "communiqué de presse") '
        f'2022..2026'
    )

    try:
        # Search Official
        res_official = tavily_client.search(
            query=query_official, 
            search_depth="advanced", 
            max_results=10)
        
        # Search Unofficial
        # Advanced depth is crucial when searching open web to extract clean text from news articles
        res_unofficial = tavily_client.search(
            query=query_unofficial, 
            search_depth="advanced", 
            max_results=10 # Increased to catch press releases
        )

        # 3. Save
        entry = {
            "id": record_id,
            "finess": finess,
            "hospital_name": name,
            "official_data": res_official.get('results', []),
            "unofficial_data": res_unofficial.get('results', []),
            "retrieved_at": time.strftime("%Y-%m-%d %H:%M:%S")
        }

        with open(output_path, "a", encoding="utf-8") as f:
            f.write(json.dumps(entry) + "\n")
        
        print(f"✅ Deep Search Complete: {name}")
        time.sleep(1)

    except Exception as e:
        print(f"❌ Error with {name}: {str(e)}")
print(f"Deep context dataset saved in: {output_path}")

✅ Deep Search Complete: CH DE FLEYRIAT
✅ Deep Search Complete: CH BUGEY SUD
✅ Deep Search Complete: CH DU HAUT BUGEY - GEOVREISSET
✅ Deep Search Complete: CLINIQUE CONVERT
✅ Deep Search Complete: HOPITAL PRIVE D AMBERIEU
✅ Deep Search Complete: CH SAINT-QUENTIN
✅ Deep Search Complete: CH SAINT-QUENTIN
✅ Deep Search Complete: CH LAON
✅ Deep Search Complete: CH LAON
✅ Deep Search Complete: CH SOISSONS
✅ Deep Search Complete: CH SOISSONS
✅ Deep Search Complete: CH CHAUNY
✅ Deep Search Complete: CH CHATEAU-THIERRY
✅ Deep Search Complete: CH HIRSON
✅ Deep Search Complete: HOPITAL PRIVE SAINT CLAUDE
✅ Deep Search Complete: CH DE MOULINS
✅ Deep Search Complete: CH DE MONTLUCON
✅ Deep Search Complete: CH JACQUES LACARIN VICHY
✅ Deep Search Complete: CHI DE MANOSQUE LOUIS RAFFALLI
✅ Deep Search Complete: CHI DES ALPES DU SUD SITE DE SISTERON
✅ Deep Search Complete: CENTRE HOSPITALIER DE DIGNE LES BAINS
✅ Deep Search Complete: CH DES ESCARTONS DE BRIANCON
✅ Deep Search Complete: CENTRE HOSPITALI

# Broad Instructions

In [87]:
df_to_extract = df_unique.head(2)

In [86]:
# 1. Setup
# Define the output jsonl file
output_path_2 = "/Data/anahi_reyes/EDCD_data/raw_edcd_database_2.jsonl"
folder_path = os.path.dirname(output_path)

# Create folder if it does not exist
os.makedirs(folder_path, exist_ok=True)
print(f"Verified directory: {output_path_2}")

Verified directory: /Data/anahi_reyes/EDCD_data/raw_edcd_database_2.jsonl


In [ ]:
# 2. Main Loop
for index, row in df_to_extract.iterrows():
    finess = row["finess"]
    name = row["nom_etab"]
    keywords_nom_etab = row["keywords_nom_etab"]
    nom_etab_long = row["nom_etab_long"]
    ville = row["ville"]
    
    # Keep it short — use keywords 
    query = f'("{name}" OR "{keywords_nom_etab}") urgences fermeture régulation accès régulé'

    try:
        search_official = tavily_client.search(
        query=query,
        search_depth="advanced",
        topic="general",  # includes ALL source types
        chunks_per_source=5,
        max_results=5,
        include_domains=["ars.sante.fr", "sante.gouv.fr"],
        country="france",
        start_date="2022-06-01",
        end_date="2025-12-31"
    )
        
        # Search Unofficial
        search_all = tavily_client.search(
        query=query,
        search_depth="advanced",
        topic="general",  
        chunks_per_source=5,
        #include_raw_content=True,
        max_results=5,
        exclude_domains=["ars.sante.fr", "sante.gouv.fr"],
        country="france",
        start_date="2022-06-01",
        end_date="2025-12-31"
    )

        # 3. Save
        entry = {
            "finess": finess,
            "hospital_name": name,
            "nom_etab_long": nom_etab_long,
            "keywords_nom_etab": keywords_nom_etab,
            "official_data": search_official.get('results', []),
            "unofficial_data": search_all.get('results', []),
            "retrieved_at": time.strftime("%Y-%m-%d %H:%M:%S")

        }

        with open(output_path, "a", encoding="utf-8") as f:
            f.write(json.dumps(entry) + "\n")
        
        print(f"✅ Deep Search Complete: {name}")
        time.sleep(1)

    except Exception as e:
        print(f"❌ Error with {name}: {str(e)}")
print(f"Deep context dataset saved in: {output_path_2}")

✅ Deep Search Complete: CH DE FLEYRIAT
✅ Deep Search Complete: CH BUGEY SUD
Deep context dataset saved in: /Data/anahi_reyes/EDCD_data/raw_edcd_database_2.jsonl


: 